<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 6 · Agregar con condiciones
**Excel Corporativo · 12 Claves** — SUMAR.SI.CONJUNTO y sus hermanas · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/06_agregar_con_condiciones.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-06)

**Al terminar podrás:**
1. Ver el informe mensual sede × canal que se arma con una sola fórmula.
2. Comprobar que el informe cuadra con una tabla dinámica.
3. Comparar lo que responden SUMAR, CONTAR y PROMEDIO con las mismas condiciones.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · El informe mensual de formato fijo
La hoja **Informe_Mensual** de `Andina_listo_para_dinamicas.xlsx` tiene las sedes en filas, los canales en columnas y el año y el mes en dos celdas (G1 y G2). Cada celda amarilla lleva la misma fórmula:

`=SUMAR.SI.CONJUNTO(tVentas[subtotal];tVentas[ciudad];$A5;tVentas[canal];B$4;tVentas[anio];$G$1;tVentas[mes];$G$2;tVentas[estado];"<>Anulada")`

Aquí ves ese mismo informe. Elige el año y el mes en la lista de la derecha, como si los escribieras en G1 y G2.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto vendió cada sede por cada canal en el mes elegido? |
> | **Qué se grafica** | Mapa de calor: sedes × canales, en millones de pesos |
> | **Por qué esta forma** | Es una tabla de doble entrada: el color guía la vista hacia las celdas grandes sin perder el número exacto. |
> | **Qué se descartó** | Barras agrupadas: con 18 barras la comparación se vuelve difícil de leer. |
> | **Cómo se lee** | Las filas oscuras son las sedes grandes; compara los canales dentro de cada fila. |
> | **En Excel 365** | Hoja Informe_Mensual: la fórmula de B5 copiada a B5:D10, más totales en la columna E y la fila 11. |

In [ ]:
# @title Gráfico 1 · Informe mensual sede × canal
anio = 2025  # @param [2024, 2025]
mes = 12  # @param {type:"slider", min:1, max:12, step:1}
periodo = validas[(validas.anio == anio) & (validas.mes == mes)]
informe = (periodo.pivot_table(index="ciudad", columns="canal", values="subtotal", aggfunc="sum", fill_value=0)
           .reindex(index=sedes.ciudad, columns=["Presencial", "Televentas", "E-commerce"], fill_value=0))
z = informe / 1e6
fig = go.Figure(go.Heatmap(z=z.values, x=z.columns, y=z.index, colorscale=[[0, "#EEF2FF"], [1, AZUL]], showscale=False,
                           text=[[f"{v:,.0f}".replace(",", ".") for v in fila] for fila in z.values], texttemplate="%{text}"))
fig.update_layout(title=f"Venta neta de {MESES[mes - 1].lower()} {anio}: {millones(informe.values.sum())}",
                  yaxis=dict(autorange="reversed"), height=380, xaxis_side="top")
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿El informe cuadra con una tabla dinámica? |
> | **Qué se grafica** | Dos barras: total del informe (fórmulas) y total de la dinámica · Tercera barra: lo que daría sin el criterio de estado |
> | **Por qué esta forma** | La comprobación es una igualdad: dos barras iguales se reconocen al instante, y la tercera muestra el error típico. |
> | **Qué se descartó** | Mostrar solo un ✓: no se ve qué pasa cuando falta el criterio. |
> | **Cómo se lee** | Las dos primeras barras deben medir lo mismo. La tercera es más larga porque suma facturas anuladas. |
> | **En Excel 365** | Tabla dinámica con anio y mes en Filtros y estado distinto de Anulada; su total general debe ser igual a E11. |

In [ ]:
# @title Gráfico 2 · Cuadre contra la tabla dinámica
total_formulas = informe.values.sum()
total_dinamica = periodo["subtotal"].sum()
sin_estado = df[(df.anio == anio) & (df.mes == mes)]["subtotal"].sum()
comp = pd.Series({"Informe con fórmulas": total_formulas, "Tabla dinámica": total_dinamica,
                  "Fórmula sin criterio de estado": sin_estado}) / 1e6
fig = go.Figure(go.Bar(x=comp.values, y=comp.index, orientation="h", marker_color=[AZUL, TEAL, MAGENTA],
                       text=[f"$ {v:,.0f} M".replace(",", ".") for v in comp.values], textposition="outside", cliponaxis=False))
cuadra = abs(total_formulas - total_dinamica) < 1
fig.update_layout(title=("Cuadra" if cuadra else "No cuadra") + f": sin el criterio de estado se suman {millones(sin_estado - total_dinamica)} de anuladas",
                  xaxis_title=MM, height=280, yaxis=dict(autorange="reversed"))
dar_aire(fig, comp.max())
fig.show()

## 2 · Las mismas condiciones, tres preguntas distintas
Con los mismos criterios (sede, año, mes, sin anuladas):
* **SUMAR.SI.CONJUNTO** responde *¿cuánto se vendió?*
* **CONTAR.SI.CONJUNTO** responde *¿cuántas líneas de venta hubo?* (no lleva rango de suma)
* **PROMEDIO.SI.CONJUNTO** responde *¿de cuánto es una línea típica?*

Una sede puede vender mucho por volumen (muchas líneas) o por valor (líneas grandes). Solo viendo las tres se sabe cuál es el caso.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cada sede vende por volumen o por valor? |
> | **Qué se grafica** | Tres paneles de barras por sede: suma, conteo y promedio por línea |
> | **Por qué esta forma** | Paneles con el mismo orden de sedes permiten comparar las tres medidas fila por fila. |
> | **Qué se descartó** | Un solo gráfico con tres medidas: tienen escalas distintas (pesos, líneas, pesos por línea). |
> | **Cómo se lee** | Si una sede baja en el conteo pero sube en el promedio, vende menos veces pero más grande. |
> | **En Excel 365** | En tres columnas nuevas: =SUMAR.SI.CONJUNTO(…), =CONTAR.SI.CONJUNTO(…) y =PROMEDIO.SI.CONJUNTO(…), con los mismos criterios. |

In [ ]:
# @title Gráfico 3 · Suma, conteo y promedio con las mismas condiciones
from plotly.subplots import make_subplots
por_sede = periodo.groupby("ciudad")["subtotal"].agg(["sum", "count", "mean"]).reindex(sedes.ciudad).fillna(0)
orden = por_sede.sort_values("sum").index
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=["SUMAR: millones", "CONTAR: líneas", "PROMEDIO: miles por línea"])
for i, (col, escala, color) in enumerate([("sum", 1e6, AZUL), ("count", 1, TEAL), ("mean", 1e3, MAGENTA)], start=1):
    s = por_sede.loc[orden, col] / escala
    fig.add_bar(x=s.values, y=list(orden), orientation="h", marker_color=color, showlegend=False,
                text=[f"{v:,.0f}".replace(",", ".") for v in s.values], textposition="inside", row=1, col=i)
mayor_prom = por_sede["mean"].idxmax()
fig.update_layout(title=f"{MESES[mes - 1]} {anio}: la línea promedio más alta está en {mayor_prom}", height=360)
fig.show()

## 3 · Criterios de comparación

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuántas facturas superan un monto? |
> | **Qué se grafica** | Barras por sede: facturas por encima del límite elegido, en todo el año |
> | **Por qué esta forma** | Pocas categorías y una medida: la barra es la forma más clara. |
> | **Qué se descartó** | Un histograma: responde otra pregunta (cómo se distribuyen los montos). |
> | **Cómo se lee** | Mueve el límite y observa qué sedes concentran las facturas grandes. |
> | **En Excel 365** | =CONTAR.SI.CONJUNTO(tFacturas[total];">"&$G$3;tFacturas[ciudad];$A5), con el límite en G3. |

In [ ]:
# @title Gráfico 4 · Facturas por encima de un límite
limite = 20000000  # @param [5000000, 10000000, 20000000, 50000000]
facturas = validas[validas.anio == anio].groupby(["id_factura", "ciudad"], as_index=False)["subtotal"].sum()
grandes = facturas[facturas.subtotal > limite].groupby("ciudad").size().reindex(sedes.ciudad, fill_value=0).sort_values()
fig = go.Figure(go.Bar(x=grandes.values, y=grandes.index, orientation="h", marker_color=AZUL,
                       text=[miles(v) for v in grandes.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{miles(grandes.sum())} facturas de {anio} superan {pesos(limite)} (criterio \">\"&G3)",
                  xaxis_title="Facturas", height=320)
dar_aire(fig, max(grandes.max(), 1))
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · En diciembre de 2025, ¿qué combinación sede · canal vendió más?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Bogotá · Presencial", "Bogotá · Televentas", "Medellín · Presencial", "Cali · Televentas"]
revisar(respuesta, (lambda s: f"{s.index[0][0]} · {s.index[0][1]}")(validas[(validas.anio == 2025) & (validas.mes == 12)].groupby(["ciudad", "canal"])["subtotal"].sum().sort_values(ascending=False)), """Es la celda más oscura del mapa de calor con año 2025 y mes 12.""")

In [ ]:
# @title Reto 2 · En el criterio de estado, ¿qué texto va entre comillas para excluir las facturas anuladas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Anulada", "<>Anulada", "=Anulada", "No Anulada"]
revisar(respuesta, "<>Anulada", """<> significa «distinto de». Como lleva un operador, todo va entre comillas: "<>Anulada".""")

In [ ]:
# @title Reto 3 · Una sede tiene menos líneas que otra pero una venta parecida. ¿Qué se puede concluir?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Que vende líneas de mayor valor", "Que tiene más facturas anuladas", "Que la fórmula está mal", "Nada"]
revisar(respuesta, "Que vende líneas de mayor valor", """Si la suma es parecida con menos líneas, el promedio por línea es mayor: vende por valor, no por volumen.""")

## Apóyate en IA
Para pasar de una pregunta de negocio a la fórmula:

```text
Tengo en Excel 365 una Tabla tVentas (ciudad, canal, anio, mes, estado, subtotal).
Quiero la venta neta por sede (A5:A10) y canal (B4:D4) del año G1 y el mes G2, sin anuladas.
Escribe la fórmula de B5 en español (separador ;) para copiarla a toda la matriz, y explica cada $.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Una fórmula con criterios en celdas, copiada a toda la matriz, arma un informe que se actualiza solo.
* Cuadra siempre contra una tabla dinámica: un criterio olvidado no muestra error.
* SUMAR, CONTAR y PROMEDIO con las mismas condiciones cuentan historias distintas.

**Siguiente:** [Clave 7 · Tablas dinámicas](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-07) · [Cuaderno 07](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/07_tablas_dinamicas.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>